In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import pandas as pd
import requests

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/live-vizier-tsv-check')
OUT.mkdir(parents=True, exist_ok=True)
TARGETS = ['30317282', '11521787']
URL = 'https://vizier.cfa.harvard.edu/viz-bin/asu-tsv'

def norm_tic(value):
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null'}:
        return ''
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError, OverflowError):
        pass
    return text

found = []
for path in INPUT.rglob('features.csv'):
    try:
        cols = pd.read_csv(path, nrows=0).columns
        lower = {str(c).strip().lower(): c for c in cols}
        id_col = next((lower[k] for k in ('source_id', 'tic', 'tic_id', 'ticid') if k in lower), None)
        ra_col = next((lower[k] for k in ('ra_deg', 'ra', 'ra_icrs', 'ra_icrs_deg') if k in lower), None)
        dec_col = next((lower[k] for k in ('dec_deg', 'dec', 'de_icrs', 'dec_icrs_deg') if k in lower), None)
        if id_col is None or ra_col is None or dec_col is None:
            continue
        frame = pd.read_csv(path, usecols=[id_col, ra_col, dec_col], dtype={id_col: 'string'}, low_memory=False)
        frame['_tic'] = frame[id_col].map(norm_tic)
        frame[ra_col] = pd.to_numeric(frame[ra_col], errors='coerce')
        frame[dec_col] = pd.to_numeric(frame[dec_col], errors='coerce')
        frame = frame[frame['_tic'].isin(TARGETS) & frame[ra_col].between(0, 360) & frame[dec_col].between(-90, 90)]
        if not frame.empty:
            found.append((len(frame), path, frame, ra_col, dec_col))
    except Exception as exc:
        print('[SKIP]', path, type(exc).__name__, str(exc)[:160])
if not found:
    raise FileNotFoundError('No attached features.csv with target TIC IDs and RA/Dec. Attach Cosmic Swan ML Day1 output.')
_, feature_path, coords, RA_COL, DEC_COL = max(found, key=lambda item: (item[0], str(item[1])))
print('[PASS] Coordinates:', feature_path, '| matching rows:', len(coords))

results = []
for tic in TARGETS:
    sub = coords[coords['_tic'].eq(tic)]
    if sub.empty:
        results.append({'source_id': tic, 'status': 'no_coordinates'})
        print('[FAIL] No coordinates for TIC', tic)
        continue
    row = sub.iloc[0]
    ra, dec = float(row[RA_COL]), float(row[DEC_COL])
    params = {
        '-source': 'I/355/gaiadr3',
        '-c.ra': f'{ra:.8f}',
        '-c.dec': f'{dec:.8f}',
        '-c.rs': '5',
        '-out': 'Source,RA_ICRS,DE_ICRS,Gmag',
        '-out.max': '5',
    }
    started = time.time()
    try:
        response = requests.get(
            URL, params=params,
            headers={'User-Agent': 'CosmicSwanLiveCatalogCheck/1.0'},
            timeout=(10, 30),
        )
        text = response.text
        raw_path = OUT / f'TIC_{tic}_vizier_live.tsv'
        raw_path.write_text(text, encoding='utf-8', errors='replace')
        has_vizier_header = 'Source' in text and ('I/355/gaiadr3' in text.lower() or 'gaiadr3' in text.lower())
        success = response.status_code == 200 and has_vizier_header and '<html' not in text[:500].lower()
        result = {
            'source_id': tic,
            'status': 'query_succeeded' if success else 'unexpected_response',
            'mirror': 'vizier.cfa.harvard.edu',
            'endpoint': '/viz-bin/asu-tsv',
            'catalog': 'I/355/gaiadr3',
            'http_status': response.status_code,
            'ra_deg': ra,
            'dec_deg': dec,
            'seconds': round(time.time()-started, 2),
            'response_file': raw_path.name,
            'response_preview': '\n'.join(text.splitlines()[:12]),
            'response_chars': len(text),
        }
        results.append(result)
        print(f"[{'PASS' if success else 'FAIL'}] VizieR TSV TIC {tic}: HTTP {response.status_code}, {len(text)} chars, {result['seconds']} sec")
    except Exception as exc:
        result = {
            'source_id': tic,
            'status': 'query_failed',
            'mirror': 'vizier.cfa.harvard.edu',
            'endpoint': '/viz-bin/asu-tsv',
            'catalog': 'I/355/gaiadr3',
            'ra_deg': ra,
            'dec_deg': dec,
            'seconds': round(time.time()-started, 2),
            'error': f'{type(exc).__name__}: {str(exc)[:400]}',
        }
        results.append(result)
        print(f"[FAIL] VizieR TSV TIC {tic}: {result['error']}")

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'feature_file': str(feature_path),
    'purpose': 'Direct read-only VizieR ASU TSV live check; no Gemini key used.',
    'targets': results,
    'interpretation': 'A query_succeeded status means the live VizieR TSV endpoint returned a Gaia DR3 response. Zero matching rows, if any, is not a catalog-wide non-detection.'
}
summary_path = OUT / 'live_vizier_tsv_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.DataFrame([{k: v for k, v in r.items() if k not in {'response_preview'}} for r in results]).to_csv(OUT / 'live_vizier_tsv_summary.csv', index=False)
print('\nSaved:', summary_path)
print(pd.DataFrame([{'TIC': r.get('source_id'), 'status': r.get('status'), 'HTTP': r.get('http_status'), 'seconds': r.get('seconds'), 'error': r.get('error', '')} for r in results]).to_string(index=False))
